# Spark ML Pipeline for Titanic Survival Classification

The goal is to predict the survival of passagers on board titanic, and to use the pipeline and feature engineering tools of Spark ML. A description of the dataset can be found at the [kaggle website](https://www.kaggle.com/c/titanic/data).

The workflow focuses on:

- Constructing a full Spark ML pipeline with feature engineering
- Using Spark SQL / DataFrames to maintain scalability and clean transformations
- Ensuring idempotency, so the notebook can be run repeatedly without errors
- Demonstrating how Spark ML handles preprocessing, vectorization, and model training in a unified pipeline

This project was a part of an MSBA course on Big Data Analytics as a practical introduction to Spark ML’s pipeline architecture while solving a classic classification problem.

## Dataset Uploading

### 1. Download and load data
- Create a Unity Catalog Volume `homework3` under the schema `spark` of the catalog `msbabigdata` (recreate them if not exists)
- Download [`titanic.csv`](https://idsdl.csom.umn.edu/c/share/msba6330/titanic.csv)
- Then, load the data into the Unity Catalog Volume `homework3`.
- Create a Spark DataFrame `titanic` using the uploaded csv file.

In [0]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.homework3;

In [0]:
path = "/Volumes/msbabigdata/spark/homework3"
titanic = spark.read.csv(path + '/titanic.csv', header=True, inferSchema=True)

### 2. Verify the DataFrame by

- Displaying the schema
- Displaying the first 5 rows of data
- Counting the number of rows

In [0]:
# Display Schema
titanic.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [0]:
# Display first 5 rows
titanic.limit(5).display()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S


In [0]:
# Display Row Count
titanic.count()

891

### 3. Split the DataFrame into Training and Testing Sets

- Randomly split the DataFrame into `train` (70%) and `test` (30%), using 1234 as the random seed.
- Print the row count for each DataFrame.

In [0]:
# 70-30% train-test split
train, test = titanic.randomSplit([0.7, 0.3], seed=1234)

In [0]:
train.count()

603

In [0]:
test.count()

288

### 4. Persist the `train` and `test` DataFrames as Spark tables in the database `titanic`

- Create the database `titanic` if it does not exist
- The table names will be `train` and `test`
- Keep in mind that all steps need to be idempotent.

In [0]:
%sql
-- Create database titanic
create database if not exists titanic

In [0]:
# Create tables titanic.train and titanic.test
train.write.mode("overwrite").saveAsTable("titanic.train")
test.write.mode("overwrite").saveAsTable("titanic.test")

## Feature Engineering

The goal of this step is to perform feature engineering to prepare for ML models. We want to do it in a way that's repeatable—so that feature engineering steps can be codified in a pipeline and applied for both model training and inference.

We will focus on:
- converting data types
- selecting columns
- dealing with missing values
- handling text and categorical columns
- assembling the feature vector

### 5. Create SQLTransformer to select given features and convert numerical columns into double
Only include the features and label (`Survived`) columns that are plausible and will be used:
  - Survived
  - Pclass
  - Age
  - SibSp
  - Parch
  - Sex
  - Fare
  - Embarked

In [0]:
# Using SQLTransformer to select given features and convert numerical columns into double
from pyspark.ml.feature import SQLTransformer

st_cast = SQLTransformer(
    statement="""
        SELECT
            CAST(Survived AS DOUBLE) AS Survived,
            CAST(Pclass AS DOUBLE) AS Pclass,
            CAST(Age AS DOUBLE) AS Age,
            CAST(SibSp AS DOUBLE) AS SibSp,
            CAST(Parch AS DOUBLE) AS Parch,
            Sex,
            CAST(Fare AS DOUBLE) AS Fare,
            Embarked
        FROM __THIS__
    """
)
train_cast = st_cast.transform(train)
train_cast.limit(10).display()
train_cast.printSchema()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked
0.0,3.0,22.0,1.0,0.0,male,7.25,S
1.0,1.0,38.0,1.0,0.0,female,71.2833,C
1.0,3.0,26.0,0.0,0.0,female,7.925,S
1.0,1.0,35.0,1.0,0.0,female,53.1,S
0.0,3.0,35.0,0.0,0.0,male,8.05,S
0.0,3.0,null,0.0,0.0,male,8.4583,Q
0.0,1.0,54.0,0.0,0.0,male,51.8625,S
0.0,3.0,2.0,3.0,1.0,male,21.075,S
1.0,3.0,27.0,0.0,2.0,female,11.1333,S
1.0,3.0,4.0,1.0,1.0,female,16.7,S


root
 |-- Survived: double (nullable = true)
 |-- Pclass: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: double (nullable = true)
 |-- Parch: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Embarked: string (nullable = true)



### 6. Handling Missing Ages

We notice that there are missing values in the `Age` and `Embarked` columns. We decide to use a mean replacement strategy for missing ages and a drop-rows strategy for missing `Embarked`. 

- First, use DataFrame.describe() to obtain the mean age.
- Then create a new DataFrame `train_missing` after you:
  - Eliminate the rows where `Embarked` is NULL.
  - Create a new `AgeFilled` column that is a version of `Age` with null values replaced by the mean age obtained in the previous step.
  - Create a new DOUBLE indicator column **AgeNA** to denote whether the `Age` value was missing (1.0 indicates missing, 0.0 indicates not missing).

In [0]:
# Finding mean age
train.describe("Age").display()

summary,Age
count,488
mean,30.02340163934426
stddev,14.06413565095182
min,0.42
max,74.0


In [0]:
age_mean = train.describe("Age").collect()[1]["Age"]


In [0]:
# Using a SQL transformer for selecting non null Embarked rows and filling null Age with mean age
st_missing = SQLTransformer(
    statement=f"""
        SELECT
            Survived,
            Pclass,
            Age,
            SibSp,
            Parch,
            Sex,
            Fare,
            Embarked,
            CASE WHEN Age IS NULL THEN {age_mean} ELSE Age END AS AgeFilled,
            CASE WHEN Age IS NULL THEN 1.0 ELSE 0.0 END AS AgeNA
        FROM __THIS__
        WHERE Embarked IS NOT NULL
    """
)
train_missing = st_missing.transform(train_cast)
train_missing.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA
0.0,3.0,22.0,1.0,0.0,male,7.25,S,22.0,0.0
1.0,1.0,38.0,1.0,0.0,female,71.2833,C,38.0,0.0
1.0,3.0,26.0,0.0,0.0,female,7.925,S,26.0,0.0
1.0,1.0,35.0,1.0,0.0,female,53.1,S,35.0,0.0
0.0,3.0,35.0,0.0,0.0,male,8.05,S,35.0,0.0
0.0,3.0,null,0.0,0.0,male,8.4583,Q,30.02340163934426,1.0
0.0,1.0,54.0,0.0,0.0,male,51.8625,S,54.0,0.0
0.0,3.0,2.0,3.0,1.0,male,21.075,S,2.0,0.0
1.0,3.0,27.0,0.0,2.0,female,11.1333,S,27.0,0.0
1.0,3.0,4.0,1.0,1.0,female,16.7,S,4.0,0.0



### 7. Index and Encode string/categorical columns

We first index string columns using StringIndexer (so you get values such as 0, 1, 2, ...), then use OneHotEncoder to convert categorical values into binary dummies.

- Create a StringIndexer `si` for indexing the two string features `Sex` and `Embarked` (the resultant columns should have a postfix of `_idx`).
- Then create a OneHotEncoder `ohe` for the indexed string columns (the resulting columns should have a postfix of `_ohe`).

In [0]:
# Convert categorical string values into numeric index labels
from pyspark.ml.feature import StringIndexer

si = StringIndexer(
    inputCols=["Sex", "Embarked"],
    outputCols=["Sex_idx", "Embarked_idx"],
    handleInvalid="keep"
)

si_model  = si.fit(train_missing)
train_si  = si_model.transform(train_missing)

train_si.display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx
0.0,3.0,22.0,1.0,0.0,male,7.25,S,22.0,0.0,0.0,0.0
1.0,1.0,38.0,1.0,0.0,female,71.2833,C,38.0,0.0,1.0,1.0
1.0,3.0,26.0,0.0,0.0,female,7.925,S,26.0,0.0,1.0,0.0
1.0,1.0,35.0,1.0,0.0,female,53.1,S,35.0,0.0,1.0,0.0
0.0,3.0,35.0,0.0,0.0,male,8.05,S,35.0,0.0,0.0,0.0
0.0,3.0,null,0.0,0.0,male,8.4583,Q,30.02340163934426,1.0,0.0,2.0
0.0,1.0,54.0,0.0,0.0,male,51.8625,S,54.0,0.0,0.0,0.0
0.0,3.0,2.0,3.0,1.0,male,21.075,S,2.0,0.0,0.0,0.0
1.0,3.0,27.0,0.0,2.0,female,11.1333,S,27.0,0.0,1.0,0.0
1.0,3.0,4.0,1.0,1.0,female,16.7,S,4.0,0.0,1.0,0.0


In [0]:
# Converts indexed categories into binary vectors
from pyspark.ml.feature import OneHotEncoder

ohe = OneHotEncoder(
    inputCols=["Sex_Idx", "Embarked_Idx"],
    outputCols=["Sex_ohe", "Embarked_ohe"]
)

train_ohe = ohe.fit(train_si).transform(train_si)
train_ohe.display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe
0.0,3.0,22.0,1.0,0.0,male,7.25,S,22.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
1.0,1.0,38.0,1.0,0.0,female,71.2833,C,38.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}"
1.0,3.0,26.0,0.0,0.0,female,7.925,S,26.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
1.0,1.0,35.0,1.0,0.0,female,53.1,S,35.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
0.0,3.0,35.0,0.0,0.0,male,8.05,S,35.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
0.0,3.0,null,0.0,0.0,male,8.4583,Q,30.02340163934426,1.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}"
0.0,1.0,54.0,0.0,0.0,male,51.8625,S,54.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
0.0,3.0,2.0,3.0,1.0,male,21.075,S,2.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
1.0,3.0,27.0,0.0,2.0,female,11.1333,S,27.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"
1.0,3.0,4.0,1.0,1.0,female,16.7,S,4.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}"


### 8. Assemble feature columns into a feature vector (for use with the pipeline)

- For the `Age` column, include both `AgeFilled` and `AgeNA`.
- For `Sex` and `Embarked`, include only the encoded versions produced by OneHotEncoder.

In [0]:
# # Assemble all feature columns into a single 'features' vector
from pyspark.ml.feature import VectorAssembler

va = VectorAssembler(
    inputCols=[
        "Pclass",
        "AgeFilled",
        "AgeNA",
        "SibSp",
        "Parch",
        "Fare",
        "Sex_ohe",
        "Embarked_ohe"
    ],
    outputCol="features"
)

train_va = va.transform(train_ohe)
train_va.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe,features
0.0,3.0,22.0,1.0,0.0,male,7.25,S,22.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""22.0"",""0.0"",""1.0"",""0.0"",""7.25"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}"
1.0,1.0,38.0,1.0,0.0,female,71.2833,C,38.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""38.0"",""0.0"",""1.0"",""0.0"",""71.2833"",""0.0"",""1.0"",""0.0"",""1.0"",""0.0""]}"
1.0,3.0,26.0,0.0,0.0,female,7.925,S,26.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""7"",""8""],""values"":[""3.0"",""26.0"",""7.925"",""1.0"",""1.0""]}"
1.0,1.0,35.0,1.0,0.0,female,53.1,S,35.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""35.0"",""0.0"",""1.0"",""0.0"",""53.1"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0""]}"
0.0,3.0,35.0,0.0,0.0,male,8.05,S,35.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""6"",""8""],""values"":[""3.0"",""35.0"",""8.05"",""1.0"",""1.0""]}"
0.0,3.0,null,0.0,0.0,male,8.4583,Q,30.02340163934426,1.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""8.4583"",""1.0"",""0.0"",""0.0"",""0.0"",""1.0""]}"
0.0,1.0,54.0,0.0,0.0,male,51.8625,S,54.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""6"",""8""],""values"":[""1.0"",""54.0"",""51.8625"",""1.0"",""1.0""]}"
0.0,3.0,2.0,3.0,1.0,male,21.075,S,2.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""2.0"",""0.0"",""3.0"",""1.0"",""21.075"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}"
1.0,3.0,27.0,0.0,2.0,female,11.1333,S,27.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""27.0"",""0.0"",""0.0"",""2.0"",""11.1333"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0""]}"
1.0,3.0,4.0,1.0,1.0,female,16.7,S,4.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""4.0"",""0.0"",""1.0"",""1.0"",""16.7"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0""]}"


## Models and ML Pipelines

### 9. Create the LogisticRegression model (for use with the pipeline)
Create the LogisticRegression estimator `lr` to predict `Survived` using our assembled feature vector.

In [0]:
# Logistic regression esitimator and prediction using training data from vector assembler
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    labelCol="Survived",
    featuresCol="features"
)
lr_model = lr.fit(train_va)
df_pred = lr_model.transform(train_va)
df_pred.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe,features,rawPrediction,probability,prediction
0.0,3.0,22.0,1.0,0.0,male,7.25,S,22.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""22.0"",""0.0"",""1.0"",""0.0"",""7.25"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.6699526696354274"",""-2.6699526696354274""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9352301644294506"",""0.06476983557054938""]}",0.0
1.0,1.0,38.0,1.0,0.0,female,71.2833,C,38.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""38.0"",""0.0"",""1.0"",""0.0"",""71.2833"",""0.0"",""1.0"",""0.0"",""1.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.8976840508886936"",""2.8976840508886936""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.052268167523774674"",""0.9477318324762253""]}",1.0
1.0,3.0,26.0,0.0,0.0,female,7.925,S,26.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""7"",""8""],""values"":[""3.0"",""26.0"",""7.925"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.521546920537264"",""0.521546920537264""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3724905831318862"",""0.6275094168681138""]}",1.0
1.0,1.0,35.0,1.0,0.0,female,53.1,S,35.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""35.0"",""0.0"",""1.0"",""0.0"",""53.1"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.5331675715653823"",""2.5331675715653823""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07356547339660524"",""0.9264345266033948""]}",1.0
0.0,3.0,35.0,0.0,0.0,male,8.05,S,35.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""6"",""8""],""values"":[""3.0"",""35.0"",""8.05"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.9579346512519855"",""-2.9579346512519855""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9506371652155833"",""0.04936283478441672""]}",0.0
0.0,3.0,null,0.0,0.0,male,8.4583,Q,30.02340163934426,1.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""8.4583"",""1.0"",""0.0"",""0.0"",""0.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0150354490903464"",""-2.0150354490903464""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.882366687388524"",""0.11763331261147603""]}",0.0
0.0,1.0,54.0,0.0,0.0,male,51.8625,S,54.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""6"",""8""],""values"":[""1.0"",""54.0"",""51.8625"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"

### 10. Define the Pipeline 

Put together the feature engineering and ML pipeline `pl`.

- The first step is casting and column selection.
- The last step is logistic regression.

In [0]:
# Define ML Pipeline
from pyspark.ml import Pipeline
pl = Pipeline(stages= [st_cast, st_missing, si, ohe, va, lr])


### 11. Train the pipeline on the training set and use the trained pipeline on the test set

- Fit the pipeline on the training set `train_table` and save the result as `pl_model`.
- Use `pl_model` to obtain predictions on the test set `test_table`, saving the resulting DataFrame as `results`.

In [0]:
# Load dataframes from saved tables
train_table = spark.table("titanic.train")
test_table = spark.table("titanic.test")

In [0]:
# Fit the piepline on train set
pl_model = pl.fit(train_table)
# Apply the pipeline to the test set
results = pl_model.transform(test_table)
results.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe,features,rawPrediction,probability,prediction
1.0,2.0,14.0,1.0,0.0,female,30.0708,C,14.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""14.0"",""0.0"",""1.0"",""0.0"",""30.0708"",""0.0"",""1.0"",""0.0"",""1.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.6252521934845396"",""2.6252521934845396""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.06753080868324253"",""0.9324691913167574""]}",1.0
1.0,1.0,58.0,0.0,0.0,female,26.55,S,58.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""7"",""8""],""values"":[""1.0"",""58.0"",""26.55"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.7105328491151313"",""1.7105328491151313""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.15309461556018455"",""0.8469053844398154""]}",1.0
0.0,3.0,2.0,4.0,1.0,male,29.125,Q,2.0,0.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""2.0"",""0.0"",""4.0"",""1.0"",""29.125"",""1.0"",""0.0"",""0.0"",""0.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.119124868432057"",""-2.119124868432057""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8927481654428514"",""0.1072518345571486""]}",0.0
1.0,2.0,null,0.0,0.0,male,13.0,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""13.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.3080557476756152"",""-1.3080557476756152""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7871876302447637"",""0.21281236975523632""]}",0.0
0.0,1.0,19.0,3.0,2.0,male,263.0,S,19.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""19.0"",""0.0"",""3.0"",""2.0"",""263.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.19250651010002873"",""-0.19250651010002873""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5479785501764393"",""0.45202144982356074""]}",0.0
0.0,3.0,null,0.0,0.0,male,7.8958,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""7.8958"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.663750206611862"",""-2.663750206611862""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9348534362984393"",""0.06514656370156069""]}",0.0
1.0,1.0,null,1.0,0.0,female,146.5208,C,30.02340163934426,1.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""30.02340163934426"",""1.0"",""1.0"",""0.0"",""

### 12. Evaluate model performance

- Define a function `evaluate` that calculates and prints the area under the ROC curve (AUC), F1 score, and accuracy metrics for the provided DataFrame.  
- Use the function to evaluate the model performance on the test data.

In [0]:
# Create an evaluate function
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
def evaluate(df):
    # AUC (ROC)
    auc_eval = BinaryClassificationEvaluator(
        labelCol="Survived",
        rawPredictionCol="probability",
        metricName="areaUnderROC"
    )
    auc = auc_eval.evaluate(df)

    # F1 score
    f1_eval = MulticlassClassificationEvaluator(
        labelCol="Survived",
        predictionCol="prediction",
        metricName="f1"
    )
    f1 = f1_eval.evaluate(df)

    # Accuracy
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="Survived",
        predictionCol="prediction",
        metricName="accuracy"
    )
    acc = acc_eval.evaluate(df)

    print(f"AUC:      {auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {acc:.4f}")

# Evaluate the test results
evaluate(results)

AUC:      0.7970
F1 Score: 0.7373
Accuracy: 0.7387


### 13. Fit a Random Forest model, obtain predictions, and report the performance metrics

- Create the random forest estimator `rf`
- Create the pipeline `rf_pipeline` with the Random Forest at the end.
- Train the pipeline and save the result as `rf_pipeline_model`
- Use the pipeline model to do inference on `test_table`
- Evaluate the performance using `evaluate()`

In [0]:
# Defining a function to save the model
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(labelCol="Survived", featuresCol="features")


In [0]:
# Random Forest pipeline
rf_pipeline = Pipeline(stages=[st_cast, st_missing, si, ohe, va, rf])


In [0]:
# Fit the RF pipeline on train set
rf_pipeline_model = rf_pipeline.fit(train_table)
# Apply the RF pipeline to the test set
rf_results = rf_pipeline_model.transform(test_table)
rf_results.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe,features,rawPrediction,probability,prediction
1.0,2.0,14.0,1.0,0.0,female,30.0708,C,14.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""14.0"",""0.0"",""1.0"",""0.0"",""30.0708"",""0.0"",""1.0"",""0.0"",""1.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9763234557443204"",""19.02367654425568""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.04881617278721602"",""0.9511838272127839""]}",1.0
1.0,1.0,58.0,0.0,0.0,female,26.55,S,58.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""7"",""8""],""values"":[""1.0"",""58.0"",""26.55"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""4.595062456401147"",""15.404937543598853""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.22975312282005733"",""0.7702468771799427""]}",1.0
0.0,3.0,2.0,4.0,1.0,male,29.125,Q,2.0,0.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""2.0"",""0.0"",""4.0"",""1.0"",""29.125"",""1.0"",""0.0"",""0.0"",""0.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""17.147092047320896"",""2.852907952679106""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8573546023660447"",""0.14264539763395526""]}",0.0
1.0,2.0,null,0.0,0.0,male,13.0,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""13.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""17.919659059590607"",""2.0803409404093927""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8959829529795303"",""0.10401704702046963""]}",0.0
0.0,1.0,19.0,3.0,2.0,male,263.0,S,19.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""19.0"",""0.0"",""3.0"",""2.0"",""263.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""12.765452826155812"",""7.2345471738441915""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6382726413077905"",""0.3617273586922095""]}",0.0
0.0,3.0,null,0.0,0.0,male,7.8958,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""7.8958"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.524456711296697"",""1.475543288703304""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9262228355648349"",""0.0737771644351652""]}",0.0
1.0,1.0,null,1.0,0.0,female,146.5208,C,30.02340163934426,1.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""30.02340163934426"",""1.0"",""1.0"",""0.0"",""146.5208"

In [0]:
# Evaluate Random Forest's test results
evaluate(rf_results)

AUC:      0.8257
F1 Score: 0.7908
Accuracy: 0.7944


### 14. Tune the Random Forest Model

We consider the following hyperparameters for our Random Forest model:

- maxDepth: 5, 7
- numTrees: 20, 30

Use CrossValidator (3 folds) to find the best hyper parameter combination, using `areaUnderROC` as the metric.

In [0]:
# Create ParamGridBuilder to define hyperparameters for tuning
from pyspark.ml.tuning import ParamGridBuilder

paramGrid = (
    ParamGridBuilder()
        .addGrid(rf.maxDepth, [5, 7])
        .addGrid(rf.numTrees, [20, 30])
        .build()
)


In [0]:
# Used in Databricks serverless for setting temp path for intermediary CV results
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = path


In [0]:
# Evaluator for Cross validation with AUC as the key metric
from pyspark.ml.evaluation import BinaryClassificationEvaluator

cv_evaluator = BinaryClassificationEvaluator(
    labelCol="Survived",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

In [0]:
# Create a CrossValidator with 3 folds
from pyspark.ml.tuning import CrossValidator

cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=cv_evaluator,
    numFolds=3
)


In [0]:
# Fit Cross Validator on training data
cv_model = cv.fit(train_table)

In [0]:
# Generate CV predictions on test data
test_cv = cv_model.transform(test_table)
test_cv.limit(10).display()

Survived,Pclass,Age,SibSp,Parch,Sex,Fare,Embarked,AgeFilled,AgeNA,Sex_idx,Embarked_idx,Sex_ohe,Embarked_ohe,features,rawPrediction,probability,prediction
1.0,2.0,14.0,1.0,0.0,female,30.0708,C,14.0,0.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""14.0"",""0.0"",""1.0"",""0.0"",""30.0708"",""0.0"",""1.0"",""0.0"",""1.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.2127007938254697"",""26.78729920617452""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.10709002646084903"",""0.892909973539151""]}",1.0
1.0,1.0,58.0,0.0,0.0,female,26.55,S,58.0,0.0,1.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""11"",""indices"":[""0"",""1"",""5"",""7"",""8""],""values"":[""1.0"",""58.0"",""26.55"",""1.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.955351405976507"",""23.04464859402349""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2318450468658836"",""0.7681549531341164""]}",1.0
0.0,3.0,2.0,4.0,1.0,male,29.125,Q,2.0,0.0,0.0,2.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""2""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""2.0"",""0.0"",""4.0"",""1.0"",""29.125"",""1.0"",""0.0"",""0.0"",""0.0"",""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""25.894897049742443"",""4.105102950257559""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8631632349914147"",""0.1368367650085853""]}",0.0
1.0,2.0,null,0.0,0.0,male,13.0,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""13.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""26.499434811015885"",""3.500565188984111""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8833144937005296"",""0.11668550629947037""]}",0.0
0.0,1.0,19.0,3.0,2.0,male,263.0,S,19.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""19.0"",""0.0"",""3.0"",""2.0"",""263.0"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""13.200875164719976"",""16.799124835280022""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.44002917215733256"",""0.5599708278426674""]}",1.0
0.0,3.0,null,0.0,0.0,male,7.8958,S,30.02340163934426,1.0,0.0,0.0,"{""type"":""0"",""size"":""2"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""0""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""30.02340163934426"",""1.0"",""0.0"",""0.0"",""7.8958"",""1.0"",""0.0"",""1.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""27.355551627557073"",""2.6444483724429237""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9118517209185693"",""0.0881482790814308""]}",0.0
1.0,1.0,null,1.0,0.0,female,146.5208,C,30.02340163934426,1.0,1.0,1.0,"{""type"":""0"",""size"":""2"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""0"",""size"":""3"",""indices"":[""1""],""values"":[""1.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""30.02340163934426"",""1.0"",""1.0"",""0.0"",""146.5208"","

In [0]:
# Evaluate CV results
evaluate(test_cv)

AUC:      0.8209
F1 Score: 0.7778
Accuracy: 0.7840


### 15. Best parameters were selected by the CrossValidator

In [0]:
# Best MaxDepth value selected by CV
cv_model.bestModel.stages[5].explainParam("maxDepth")


'maxDepth: Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30]. (default: 5, current: 5)'

In [0]:
# Optimal no. of trees selected by CV
cv_model.bestModel.stages[5].explainParam("numTrees")

'numTrees: Number of trees to train (>= 1). (default: 20, current: 30)'

### 16. Persist the trained model for later use

We save the trained logistic regression model as well as the best Random Forest model for later use.
- Save them in the homework volume under the folders `lr_pipeline_model` and `rf_pipeline_best_model`, respectively.

Then, test the saved model:
- Load the `rf_pipeline_best_model` using `PipelineModel.read().load([path])`.
- Use the reloaded model to do inference on the `test_table` again.
- Evaluate the model performance to see if you get the same result.

In [0]:
from pyspark.ml import PipelineModel

lr_save_path = f"{path}/lr_pipeline_model"
rf_save_path = f"{path}/rf_pipeline_best_model"

# Save both models (overwrite for idempotency)
pl_model.write().overwrite().save(lr_save_path)
cv_model.bestModel.write().overwrite().save(rf_save_path)


In [0]:
# Reload best RF pipeline model
loaded_rf_model = PipelineModel.read().load(rf_save_path)

# Run inference on test set again
reloaded_results = loaded_rf_model.transform(test_table)

# Verify if it matches with cv_results metrics above
evaluate(reloaded_results)

AUC:      0.8209
F1 Score: 0.7778
Accuracy: 0.7840


### 17. Clean Up

Clean up the volumes, folders, tables, and database generated for the homework.

In [0]:
# Remove saved model folders from the volume
dbutils.fs.rm(f"{path}/lr_pipeline_model",      recurse=True)
dbutils.fs.rm(f"{path}/rf_pipeline_best_model", recurse=True)
dbutils.fs.rm(f"{path}/titanic.csv")
print("Volume files removed.")

Volume files removed.


In [0]:
%sql
-- Drop tables and database
DROP TABLE   IF EXISTS titanic.train;
DROP TABLE   IF EXISTS titanic.test;
DROP DATABASE IF EXISTS titanic;
DROP VOLUME IF EXISTS msbabigdata.spark.homework3;